# Old Trafford Pitch Telemetry

In [ ]:
import requests
import pandas as pd

# The live Open-Meteo API URL for Old Trafford
url = "https://api.open-meteo.com/v1/forecast?latitude=53.4631&longitude=-2.2913&hourly=temperature_2m,precipitation,wind_speed_10m&forecast_days=1"

In [ ]:
response = requests.get(url)
if response.status_code == 200:
  print("Connection successful! Status code: ", response.status_code)
  weather_forecast = response.json()
  print(weather_forecast)
else:
  print("Connection failed! Status code: ", response.status_code)

In [ ]:
hourly_forecast = weather_forecast['hourly']
hourly_forecast

In [ ]:
hourly_forecast_df = pd.DataFrame(hourly_forecast)
hourly_forecast_df.head()

In [ ]:
hourly_forecast_df['date'] = pd.to_datetime(hourly_forecast_df['time'], errors='coerce')
hourly_forecast_df.head()

In [ ]:
# 1. Filter the DataFrame for the 18:00 to 20:00 window
match_window = hourly_forecast_df[(hourly_forecast_df['date'].dt.hour >= 18) & 
                                  (hourly_forecast_df['date'].dt.hour <= 20)]

# 2. Calculate the averages
avg_temp = match_window['temperature_2m'].mean()
avg_wind = match_window['wind_speed_10m'].mean()

print(f"Match Window Temp: {avg_temp:.1f}°C")
print(f"Match Window Wind: {avg_wind:.1f} km/h")

In [ ]:
# 1. Total Precipitation
total_rain = hourly_forecast_df['precipitation'].sum()

# 2. Maximum Wind Speed
max_wind = hourly_forecast_df['wind_speed_10m'].max()

# 3. The exact hour of maximum wind using idxmax()
# idxmax() finds the index number of the highest value, which we then use to locate the date!
peak_wind_index = hourly_forecast_df['wind_speed_10m'].idxmax()
peak_wind_time = hourly_forecast_df.loc[peak_wind_index, 'date']

print(f"Total 24h Rain: {total_rain}mm")
print(f"Peak Wind: {max_wind} km/h occurring at {peak_wind_time}")

In [ ]:
# 1. Set the default state for the entire column
hourly_forecast_df['Wind_Warning'] = 'Clear'

# 2. Overwrite the rows where wind speed is strictly greater than 10.0
hourly_forecast_df.loc[hourly_forecast_df['wind_speed_10m'] > 10.0, 'Wind_Warning'] = 'High Risk'

# 3. Count the occurrences of "High Risk"
high_risk_hours = (hourly_forecast_df['Wind_Warning'] == 'High Risk').sum()

# Alternatively, using value_counts() is great for a full breakdown:
# print(hourly_forecast_df['Wind_Warning'].value_counts())

print(f"Total High Risk Hours: {high_risk_hours}")